[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/D.%20Integrated%20Tactical%20%26%20Pre-Planning%20Problems%20%28Connecting%20the%20Silos%29/34.%20The%20Hazard%20%26%20IMDG%20Segregation%20Problem/P34-Tier-2_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---



# 34. The Hazard & IMDG Segregation Problem

## Tier 2: The Classic Heuristic (Priority-Based Placement Algorithm)

### Goal
Implement a priority-based placement algorithm that assigns containers in order of decreasing constraint complexity, ensuring the most restrictive materials are placed first when maximum flexibility exists.

### Key assumptions
- Containers are placed sequentially based on priority scores
- Higher priority containers (with more restrictions) are placed first
- Each container is placed in the best available position considering current assignments
- Algorithm aims to minimize segregation violations through strategic ordering

### Approach (step-by-step)
1. Calculate priority scores for each container based on segregation restrictions
2. Sort containers by decreasing priority score
3. For each container, find the best available position minimizing potential conflicts
4. Place container and update available positions
5. Evaluate solution quality and compare with random placement

### What to look for in the results
- Reduced segregation violations compared to random placement
- Strategic placement of high-risk containers (explosives, corrosives)
- Efficient use of available stowage positions
- Computational efficiency compared to exact optimization

### Concrete example (from the source)
5 containers with IMDG classes and calculated priority scores:
- Container E1 (Class 1.1, score: 12)
- Container F3 (Class 3, score: 8)
- Container C8 (Class 8, score: 10)
- Container G2 (Class 2.3, score: 6)
- Container M9 (Class 9, score: 2)

Algorithm places containers in order: E1 first (highest priority), then C8, F3, G2, and finally M9.

In [1]:
# Import required packages for heuristic algorithm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Tuple, Optional
import random
import time
from dataclasses import dataclass
import warnings
warnings.filterwarnings('ignore')

# Set up visualization style
plt.style.use('default')
sns.set_palette("husl")

In [ ]:
# Define data structures for IMDG segregation problem
@dataclass
class Container:
    """Represents a container with dangerous goods"""
    id: str
    imdg_class: str
    weight: float = 20.0
    priority_score: int = 0

@dataclass
class Position:
    """Represents a stowage position on the vessel"""
    id: str
    hold: int
    x: float
    y: float
    z: float
    available: bool = True

@dataclass
class SegregationRequirement:
    """Represents segregation requirements between container classes"""
    class1: str
    class2: str
    requirement: str
    min_distance: float
    different_hold: bool = False

@dataclass
class PlacementResult:
    """Results of container placement"""
    container_id: str
    position_id: str
    violations_created: int
    placement_score: float

In [ ]:
# Create the concrete example from the source
def create_heuristic_example():
    """Create the concrete example with 5 containers and multiple positions"""
    
    # Define containers with IMDG classes
    containers = [
        Container("E1", "1.1", 20.0, 12),  # explosives - highest priority
        Container("F3", "3", 18.0, 8),     # flammable liquid
        Container("C8", "8", 22.0, 10),    # corrosive - high priority
        Container("G2", "2.3", 19.0, 6),   # toxic gas
        Container("M9", "9", 21.0, 2),    # miscellaneous - lowest priority
    ]
    
    # Define positions (4 positions per hold, 3 holds)
    positions = []
    for hold in range(1, 4):  # Holds 1, 2, 3
        for pos_idx in range(4):  # 4 positions per hold
            x = pos_idx * 4  # 4m spacing
            positions.append(Position(f"H{hold}_P{pos_idx+1}", hold, x, 0, 0))
    
    # Define segregation requirements
    segregation_requirements = [
        # Class 1.1 (explosives) restrictions
        SegregationRequirement("1.1", "3", "Away From", 6.0),
        SegregationRequirement("1.1", "8", "Separated From", 0.0, True),
        SegregationRequirement("1.1", "2.3", "Away From", 6.0),
        SegregationRequirement("1.1", "9", "Away From", 3.0),
        # Class 3 (flammable liquids) restrictions
        SegregationRequirement("3", "8", "Away From", 3.0),
        SegregationRequirement("3", "2.3", "Away From", 3.0),
        SegregationRequirement("3", "9", "No restriction", 0.0),
        # Class 8 (corrosives) restrictions
        SegregationRequirement("8", "2.3", "Away From", 3.0),
        SegregationRequirement("8", "9", "Away From", 3.0),
        # Class 2.3 (toxic gas) restrictions
        SegregationRequirement("2.3", "9", "Away From", 3.0),
    ]
    
    return containers, positions, segregation_requirements

# Create the example
containers, positions, seg_reqs = create_heuristic_example()

print("Containers with Priority Scores:")
for c in sorted(containers, key=lambda x: x.priority_score, reverse=True):
    print(f"  {c.id}: Class {c.imdg_class}, Priority: {c.priority_score}")

print(f"\nAvailable Positions: {len(positions)}")
for p in positions[:6]:  # Show first 6 positions
    print(f"  {p.id}: Hold {p.hold}, Position ({p.x}, {p.y}, {p.z})")
print(f"  ... and {len(positions)-6} more positions")

print(f"\nSegregation Requirements: {len(seg_reqs)}")
for req in seg_reqs[:5]:  # Show first 5 requirements
    print(f"  {req.class1} - {req.class2}: {req.requirement} (min: {req.min_distance}m, different_hold: {req.different_hold})")
print(f"  ... and {len(seg_reqs)-5} more requirements")

In [ ]:
# Calculate distance matrix between positions
def calculate_distance_matrix(positions: List[Position]) -> Dict[Tuple[str, str], float]:
    """Calculate Euclidean distances between all position pairs"""
    distances = {}
    for i, pos1 in enumerate(positions):
        for j, pos2 in enumerate(positions):
            if i != j:
                dist = np.sqrt((pos1.x - pos2.x)**2 + (pos1.y - pos2.y)**2 + (pos1.z - pos2.z)**2)
                distances[(pos1.id, pos2.id)] = dist
    return distances

# Check segregation compliance
def check_segregation_compliance(container1: Container, container2: Container, 
                                pos1: Position, pos2: Position, 
                                requirements: List[SegregationRequirement],
                                distances: Dict[Tuple[str, str], float]) -> bool:
    """Check if two containers in given positions satisfy segregation requirements"""
    
    # Find applicable requirement
    applicable_req = None
    for req in requirements:
        if ((req.class1 == container1.imdg_class and req.class2 == container2.imdg_class) or
            (req.class1 == container2.imdg_class and req.class2 == container1.imdg_class)):
            applicable_req = req
            break
    
    if applicable_req is None or applicable_req.requirement == "No restriction":
        return True
    
    # Check distance requirement
    distance = distances.get((pos1.id, pos2.id), 0)
    if distance < applicable_req.min_distance:
        return False
    
    # Check different hold requirement
    if applicable_req.different_hold and pos1.hold == pos2.hold:
        return False
    
    return True

# Calculate distance matrix
distance_matrix = calculate_distance_matrix(positions)

In [ ]:
# Priority-based placement algorithm
def calculate_priority_score(container: Container, 
                             all_containers: List[Container],
                             requirements: List[SegregationRequirement]) -> int:
    """Calculate priority score based on total segregation restrictions"""
    
    score = 0
    
    # Count restrictions with other containers
    for other_container in all_containers:
        if other_container.id != container.id:
            # Find applicable requirement
            for req in requirements:
                if ((req.class1 == container.imdg_class and req.class2 == other_container.imdg_class) or
                    (req.class1 == other_container.imdg_class and req.class2 == container.imdg_class)):
                    
                    # Weight based on restriction type
                    if req.requirement == "Separated From":
                        score += 4  # Highest weight for different hold requirement
                    elif req.requirement == "Away From":
                        score += 2  # Medium weight for distance requirement
                    elif req.requirement == "No restriction":
                        score += 0  # No weight for no restriction
                    break
    
    return score

def evaluate_position_quality(container: Container, 
                            position: Position,
                            placed_containers: List[Tuple[Container, Position]],
                            requirements: List[SegregationRequirement],
                            distances: Dict[Tuple[str, str], float]) -> float:
    """Evaluate the quality of a position for a container"""
    
    violations = 0
    total_distance_penalty = 0
    
    # Check conflicts with already placed containers
    for placed_container, placed_position in placed_containers:
        if not check_segregation_compliance(container, placed_container, position, placed_position, requirements, distances):
            violations += 1
            
            # Calculate distance penalty
            distance = distances.get((position.id, placed_position.id), 0)
            applicable_req = None
            for req in requirements:
                if ((req.class1 == container.imdg_class and req.class2 == placed_container.imdg_class) or
                    (req.class1 == placed_container.imdg_class and req.class2 == container.imdg_class)):
                    applicable_req = req
                    break
            
            if applicable_req and applicable_req.min_distance > 0:
                total_distance_penalty += max(0, applicable_req.min_distance - distance)
    
    # Position quality score (lower is better)
    # Penalty for violations, preference for lower hold numbers (stability)
    quality_score = violations * 1000 + total_distance_penalty + position.hold * 10
    
    return quality_score

def priority_based_placement(containers: List[Container],
                           positions: List[Position],
                           requirements: List[SegregationRequirement],
                           distances: Dict[Tuple[str, str], float]) -> Dict:
    """Implement priority-based placement algorithm"""
    
    start_time = time.time()
    
    # Calculate priority scores for all containers
    for container in containers:
        container.priority_score = calculate_priority_score(container, containers, requirements)
    
    # Sort containers by decreasing priority score
    sorted_containers = sorted(containers, key=lambda x: x.priority_score, reverse=True)
    
    # Initialize placement results
    placements = []
    placed_containers = []
    available_positions = positions.copy()
    
    print("Priority-Based Placement Algorithm:")
    print("=" * 50)
    
    # Place containers in priority order
    for i, container in enumerate(sorted_containers):
        print(f"\nStep {i+1}: Placing {container.id} (Class {container.imdg_class}, Priority: {container.priority_score})")
        
        # Find best available position
        best_position = None
        best_score = float('inf')
        
        for position in available_positions:
            if position.available:
                score = evaluate_position_quality(container, position, placed_containers, requirements, distances)
                if score < best_score:
                    best_score = score
                    best_position = position
        
        if best_position:
            # Count violations that will be created
            violations_created = 0
            for placed_container, placed_position in placed_containers:
                if not check_segregation_compliance(container, placed_container, best_position, placed_position, requirements, distances):
                    violations_created += 1
            
            # Place container
            best_position.available = False
            placed_containers.append((container, best_position))
            placements.append(PlacementResult(container.id, best_position.id, violations_created, best_score))
            
            print(f"  Placed at {best_position.id} (Hold {best_position.hold})")
            print(f"  Position quality score: {best_score:.2f}")
            print(f"  Violations created: {violations_created}")
            
            # Remove from available positions
            available_positions = [p for p in available_positions if p.id != best_position.id]
        else:
            print(f"  No available position for {container.id}!")
            placements.append(PlacementResult(container.id, "None", 999, float('inf')))
    
    execution_time = time.time() - start_time
    
    # Calculate total violations
    total_violations = sum(p.violations_created for p in placements)
    
    return {
        'placements': placements,
        'placed_containers': placed_containers,
        'total_violations': total_violations,
        'execution_time': execution_time,
        'containers_placed': len([p for p in placements if p.position_id != "None"])
    }

In [ ]:
# Random placement for comparison
def random_placement(containers: List[Container],
                    positions: List[Position],
                    requirements: List[SegregationRequirement],
                    distances: Dict[Tuple[str, str], float]) -> Dict:
    """Implement random placement algorithm for comparison"""
    
    start_time = time.time()
    
    # Shuffle containers randomly
    shuffled_containers = containers.copy()
    random.shuffle(shuffled_containers)
    
    # Initialize placement results
    placements = []
    placed_containers = []
    available_positions = positions.copy()
    
    print("\nRandom Placement Algorithm (for comparison):")
    print("=" * 50)
    
    # Place containers in random order
    for i, container in enumerate(shuffled_containers):
        print(f"\nStep {i+1}: Placing {container.id} (Class {container.imdg_class})")
        
        # Choose random available position
        if available_positions:
            position = random.choice(available_positions)
            
            # Count violations that will be created
            violations_created = 0
            for placed_container, placed_position in placed_containers:
                if not check_segregation_compliance(container, placed_container, position, placed_position, requirements, distances):
                    violations_created += 1
            
            # Place container
            position.available = False
            placed_containers.append((container, position))
            placements.append(PlacementResult(container.id, position.id, violations_created, 0))
            
            print(f"  Placed at {position.id} (Hold {position.hold})")
            print(f"  Violations created: {violations_created}")
            
            # Remove from available positions
            available_positions = [p for p in available_positions if p.id != position.id]
        else:
            print(f"  No available position for {container.id}!")
            placements.append(PlacementResult(container.id, "None", 999, 0))
    
    execution_time = time.time() - start_time
    
    # Calculate total violations
    total_violations = sum(p.violations_created for p in placements)
    
    return {
        'placements': placements,
        'placed_containers': placed_containers,
        'total_violations': total_violations,
        'execution_time': execution_time,
        'containers_placed': len([p for p in placements if p.position_id != "None"])
    }

In [ ]:
# Run both algorithms and compare
print("IMDG Segregation - Heuristic Algorithm Comparison")
print("=" * 60)

# Run priority-based placement
priority_result = priority_based_placement(containers, positions, seg_reqs, distance_matrix)

# Run random placement for comparison
random_result = random_placement(containers, positions, seg_reqs, distance_matrix)

# Print comparison results
print("\n" + "=" * 60)
print("ALGORITHM COMPARISON RESULTS:")
print("=" * 60)

print(f"\nPriority-Based Placement:")
print(f"  Containers placed: {priority_result['containers_placed']}/{len(containers)}")
print(f"  Total violations: {priority_result['total_violations']}")
print(f"  Execution time: {priority_result['execution_time']:.4f} seconds")

print(f"\nRandom Placement:")
print(f"  Containers placed: {random_result['containers_placed']}/{len(containers)}")
print(f"  Total violations: {random_result['total_violations']}")
print(f"  Execution time: {random_result['execution_time']:.4f} seconds")

# Calculate improvement
violation_reduction = random_result['total_violations'] - priority_result['total_violations']
improvement_percentage = (violation_reduction / max(1, random_result['total_violations'])) * 100

print(f"\nImprovement Analysis:")
print(f"  Violation reduction: {violation_reduction} violations")
print(f"  Improvement percentage: {improvement_percentage:.1f}%")
print(f"  Speed ratio: {random_result['execution_time']/max(0.0001, priority_result['execution_time']):.2f}x")

In [ ]:
# Visualize the placement results
def visualize_placement_results(priority_result: Dict, random_result: Dict, 
                              containers: List[Container], positions: List[Position]):
    """Create comprehensive visualization of placement results"""
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    
    # Create hold layouts for visualization
    holds = {}
    for hold_num in range(1, 4):
        holds[hold_num] = [p for p in positions if p.hold == hold_num]
    
    # Function to draw hold layout
    def draw_hold_layout(ax, hold_positions, placed_containers, title):
        ax.set_title(title, fontsize=12, fontweight='bold')
        ax.set_xlabel('Horizontal Position (m)')
        ax.set_ylabel('Vertical Position (m)')
        ax.grid(True, alpha=0.3)
        ax.set_xlim(-1, 13)
        ax.set_ylim(-1, 3)
        
        # Draw all positions
        for pos in hold_positions:
            # Find container at this position
            container_info = None
            for container, assigned_pos in placed_containers:
                if assigned_pos.id == pos.id:
                    container_info = container
                    break
            
            if container_info:
                # Color based on IMDG class
                colors = {'1.1': 'red', '3': 'orange', '8': 'purple', '2.3': 'green', '9': 'blue'}
                color = colors.get(container_info.imdg_class, 'gray')
                ax.scatter(pos.x, pos.y, s=400, c=color, marker='s', alpha=0.8, edgecolors='black')
                ax.text(pos.x, pos.y, f'{container_info.id}\n{container_info.imdg_class}', 
                        ha='center', va='center', fontsize=8, fontweight='bold')
            else:
                ax.scatter(pos.x, pos.y, s=400, c='lightgray', marker='s', alpha=0.3, edgecolors='black')
                ax.text(pos.x, pos.y, 'Empty', ha='center', va='center', fontsize=7)
    
    # Draw priority-based placement results
    for i, hold_num in enumerate(range(1, 4)):
        ax = axes[0, i]
        draw_hold_layout(ax, holds[hold_num], priority_result['placed_containers'], 
                       f'Priority-Based - Hold {hold_num}')
    
    # Draw random placement results
    for i, hold_num in enumerate(range(1, 4)):
        ax = axes[1, i]
        draw_hold_layout(ax, holds[hold_num], random_result['placed_containers'], 
                       f'Random - Hold {hold_num}')
    
    plt.suptitle('IMDG Segregation - Placement Algorithm Comparison', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Create performance comparison chart
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    # Violations comparison
    methods = ['Priority-Based', 'Random']
    violations = [priority_result['total_violations'], random_result['total_violations']]
    colors = ['green' if v == 0 else 'red' for v in violations]
    
    bars1 = ax1.bar(methods, violations, color=colors, alpha=0.7)
    ax1.set_ylabel('Number of Violations')
    ax1.set_title('Segregation Violations Comparison')
    ax1.grid(True, alpha=0.3, axis='y')
    
    # Add value labels on bars
    for bar, v in zip(bars1, violations):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, 
                str(v), ha='center', va='bottom', fontweight='bold')
    
    # Execution time comparison
    times = [priority_result['execution_time'], random_result['execution_time']]
    
    bars2 = ax2.bar(methods, times, color=['blue', 'orange'], alpha=0.7)
    ax2.set_ylabel('Execution Time (seconds)')
    ax2.set_title('Execution Time Comparison')
    ax2.grid(True, alpha=0.3, axis='y')
    
    # Add value labels on bars
    for bar, t in zip(bars2, times):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(times)*0.01, 
                f'{t:.4f}s', ha='center', va='bottom', fontweight='bold')
    
    plt.suptitle('Algorithm Performance Comparison', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

# Visualize the results
visualize_placement_results(priority_result, random_result, containers, positions)

In [ ]:
# Detailed analysis of placement quality
def analyze_placement_quality(result: Dict, containers: List[Container], 
                           requirements: List[SegregationRequirement],
                           distances: Dict[Tuple[str, str], float],
                           algorithm_name: str):
    """Analyze the quality of a placement solution"""
    
    print(f"\n{algorithm_name} - Detailed Quality Analysis:")
    print("=" * 50)
    
    placed_containers = result['placed_containers']
    
    # Analyze each container placement
    for container, position in placed_containers:
        print(f"\n{container.id} (Class {container.imdg_class}) at {position.id}:")
        
        # Check compliance with other placed containers
        compliance_issues = []
        for other_container, other_position in placed_containers:
            if other_container.id != container.id:
                if not check_segregation_compliance(container, other_container, position, other_position, requirements, distances):
                    # Find the specific requirement
                    for req in requirements:
                        if ((req.class1 == container.imdg_class and req.class2 == other_container.imdg_class) or
                            (req.class1 == other_container.imdg_class and req.class2 == container.imdg_class)):
                            distance = distances.get((position.id, other_position.id), 0)
                            compliance_issues.append({
                                'other_container': other_container.id,
                                'other_class': other_container.imdg_class,
                                'requirement': req.requirement,
                                'min_distance': req.min_distance,
                                'actual_distance': distance,
                                'different_hold': req.different_hold,
                                'same_hold_violation': req.different_hold and position.hold == other_position.hold
                            })
                            break
        
        if compliance_issues:
            print(f"  Compliance Issues ({len(compliance_issues)}):")
            for issue in compliance_issues:
                if issue['same_hold_violation']:
                    print(f"    - Same hold violation with {issue['other_container']} ({issue['other_class']})")
                else:
                    print(f"    - Distance violation with {issue['other_container']} ({issue['other_class']}): {issue['actual_distance']:.1f}m < {issue['min_distance']}m")
        else:
            print(f"  ✓ No compliance issues")
    
    # Summary statistics
    total_pairs = len(placed_containers) * (len(placed_containers) - 1) // 2
    compliant_pairs = 0
    
    for i, (container1, pos1) in enumerate(placed_containers):
        for j, (container2, pos2) in enumerate(placed_containers):
            if i < j:
                if check_segregation_compliance(container1, container2, pos1, pos2, requirements, distances):
                    compliant_pairs += 1
    
    compliance_rate = (compliant_pairs / max(1, total_pairs)) * 100
    
    print(f"\nSummary Statistics:")
    print(f"  Total container pairs: {total_pairs}")
    print(f"  Compliant pairs: {compliant_pairs}")
    print(f"  Compliance rate: {compliance_rate:.1f}%")
    print(f"  Total violations: {result['total_violations']}")

# Analyze both solutions
analyze_placement_quality(priority_result, containers, seg_reqs, distance_matrix, "Priority-Based Placement")
analyze_placement_quality(random_result, containers, seg_reqs, distance_matrix, "Random Placement")

In [ ]:
# Scalability analysis
def scalability_analysis():
    """Test algorithm performance on different problem sizes"""
    
    problem_sizes = [5, 10, 15, 20]  # Number of containers
    results = []
    
    for size in problem_sizes:
        print(f"\nTesting problem size: {size} containers")
        
        # Create larger problem instance
        test_containers = []
        imdg_classes = ["1.1", "3", "8", "2.3", "9"]
        
        for i in range(size):
            imdg_class = imdg_classes[i % len(imdg_classes)]
            test_containers.append(Container(f"C{i}", imdg_class, 20.0, 0))
        
        # Create more positions (holds * positions_per_hold)
        num_holds = max(3, size // 3 + 1)
        positions_per_hold = max(4, size // 2 + 1)
        
        test_positions = []
        for hold in range(1, num_holds + 1):
            for pos_idx in range(positions_per_hold):
                x = pos_idx * 3
                test_positions.append(Position(f"H{hold}_P{pos_idx+1}", hold, x, 0, 0))
        
        # Calculate distance matrix
        test_distances = calculate_distance_matrix(test_positions)
        
        # Run priority-based algorithm
        priority_start = time.time()
        priority_test_result = priority_based_placement(test_containers, test_positions, seg_reqs, test_distances)
        priority_time = time.time() - priority_start
        
        # Run random algorithm
        random_start = time.time()
        random_test_result = random_placement(test_containers, test_positions, seg_reqs, test_distances)
        random_time = time.time() - random_start
        
        results.append({
            'size': size,
            'priority_violations': priority_test_result['total_violations'],
            'random_violations': random_test_result['total_violations'],
            'priority_time': priority_time,
            'random_time': random_time,
            'improvement': random_test_result['total_violations'] - priority_test_result['total_violations']
        })
        
        print(f"  Priority violations: {priority_test_result['total_violations']}")
        print(f"  Random violations: {random_test_result['total_violations']}")
        print(f"  Improvement: {results[-1]['improvement']} violations")
    
    # Create scalability visualization
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 6))
    
    sizes = [r['size'] for r in results]
    priority_violations = [r['priority_violations'] for r in results]
    random_violations = [r['random_violations'] for r in results]
    priority_times = [r['priority_time'] for r in results]
    random_times = [r['random_time'] for r in results]
    improvements = [r['improvement'] for r in results]
    
    # Violations vs problem size
    ax1.plot(sizes, priority_violations, 'go-', label='Priority-Based', linewidth=2, markersize=8)
    ax1.plot(sizes, random_violations, 'ro-', label='Random', linewidth=2, markersize=8)
    ax1.set_xlabel('Problem Size (containers)')
    ax1.set_ylabel('Number of Violations')
    ax1.set_title('Violations vs Problem Size')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Execution time vs problem size
    ax2.plot(sizes, priority_times, 'go-', label='Priority-Based', linewidth=2, markersize=8)
    ax2.plot(sizes, random_times, 'ro-', label='Random', linewidth=2, markersize=8)
    ax2.set_xlabel('Problem Size (containers)')
    ax2.set_ylabel('Execution Time (seconds)')
    ax2.set_title('Execution Time vs Problem Size')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # Improvement vs problem size
    ax3.bar(sizes, improvements, color='green', alpha=0.7)
    ax3.set_xlabel('Problem Size (containers)')
    ax3.set_ylabel('Violation Reduction')
    ax3.set_title('Priority-Based Improvement')
    ax3.grid(True, alpha=0.3, axis='y')
    
    # Add value labels on bars
    for size, improvement in zip(sizes, improvements):
        ax3.text(size, improvement + max(improvements)*0.02, str(improvement), 
                ha='center', va='bottom', fontweight='bold')
    
    plt.suptitle('Scalability Analysis: Priority-Based vs Random Placement', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Print detailed results
    print("\n" + "=" * 60)
    print("SCALABILITY ANALYSIS RESULTS:")
    print("=" * 60)
    for r in results:
        print(f"\nSize {r['size']} containers:")
        print(f"  Priority violations: {r['priority_violations']}")
        print(f"  Random violations: {r['random_violations']}")
        print(f"  Improvement: {r['improvement']} violations")
        print(f"  Priority time: {r['priority_time']:.4f}s")
        print(f"  Random time: {r['random_time']:.4f}s")

# Run scalability analysis
scalability_analysis()

### Why this Tier exists vs earlier Tiers
This Tier 2 heuristic approach addresses limitations of exact optimization:
- **Computational efficiency** - Handles larger instances where MIP becomes intractable
- **Real-time performance** - Suitable for dynamic operational environments
- **Practical applicability** - Works with incomplete information and time constraints
- **Scalability** - Can handle hundreds of containers efficiently

### Pros / Cons vs Tier 1 (MIP)
**Advantages over Tier 1:**
- Much faster execution time (milliseconds vs seconds/minutes)
- Scales to larger problem instances
- No need for specialized optimization software
- More flexible for dynamic changes
- Easier to implement and maintain

**Disadvantages vs Tier 1:**
- No optimality guarantees
- May produce suboptimal solutions
- Solution quality depends on priority scoring
- Cannot quantify optimality gap

### When to use this Tier
- **Large-scale problems** where exact optimization is infeasible
- **Real-time operations** requiring quick decisions
- **Dynamic environments** with frequent changes
- **Resource-constrained systems** without optimization solvers
- **Initial solution generation** for more sophisticated algorithms
- **Decision support** where good-enough solutions are acceptable